# Native six-bit walk reference (tket source)

The original Notebook contains only commented code. This runnable migration uses
FlagQuantum exclusively; it does not execute or depend on the original SDK.

The source shift is modular increment. Coin=1 applies increment; coin=0 applies its inverse. The logical value is six-bit unsigned, with source array index zero as the low bit.

Target basis: CX/U3. U3 is the three-angle U gate. All explicit conversion rules preserve absolute phase.

The source calls Tket box decomposition, synthesis and an Aer backend. We report actual FlagQuantum CX/U3 costs instead of claiming the unspecified Aer/Tket basis or private pass equivalence.


## Preserve the reference convention in native Python

The historical source is commented code, now reconstructed with FlagQuantum
allocation, coherent control and modular arithmetic. Coin one increments and
coin zero decrements the six-bit position. The source name records provenance;
it does not mean Tket is installed, invoked or benchmarked by this Notebook.

Try a superposed coin at a wrap boundary and compare all amplitudes with the
independent Circuit. The output view, compiler workspace and physical converted
width are different quantities. Local synthesis and basis-conversion times
are reported separately, alongside all-column and coherent-state acceptance.


In [ ]:
from typing import Annotated

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    H,
    X,
    Output,
    QArray,
    QBit,
    QNum,
    adjoint,
    allocate,
    control,
    qfunc,
)


@qfunc
def increment(x: QNum) -> None:
    x += 1


@qfunc
def decrement(x: QNum) -> None:
    x -= 1


@qfunc
def walk(x: Output[Annotated[QNum, 6]], coin: Output[QBit]) -> None:
    allocate(x)
    allocate(coin)
    H(coin)
    control(coin == 1, lambda: increment(x))
    control(coin == 0, lambda: decrement(x))


program = synth.synthesize(walk, max_qubits=16, max_workspace=9, max_operations=20_000)
result = fq.run(program)
print(dict(program.probabilities(result, registers=("x", "coin"))))


In [ ]:
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.reference_workflows import (
    run_workflow,
)

workflow = run_workflow("tket")
print(workflow.resources)
print("synthesis_seconds", workflow.synthesis_seconds)
print("lowering_seconds", workflow.lowering_seconds)
print("local_workflow_seconds", workflow.elapsed_seconds)


In [ ]:
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.reference_circuit import (
    reference_circuit,
)
from examples.synthesis.tutorials.technology_demonstrations.classiq_paper.quantum_walk.reference_workflows import (
    logical_state,
)
import torch

expected = fq.run(reference_circuit(catalyst=False)).statevector().flatten()
for executable in (program.circuit, workflow.executable):
    torch.testing.assert_close(
        logical_state(executable, program), expected, atol=1e-11, rtol=0
    )
print("independent_full_state_check", "passed")


Source outputs are empty in version control. Tests also check every logical
input column and complex coherent inputs; this initial-state comparison alone
does not prove phase or cleanup. No original compiler timing is reproduced.
